In [2]:
import scanpy as sc
import hdf5plugin
import anndata as ad

import numpy as np
from numpy.random import choice

from scipy.stats import chi2_contingency

# Load Data:

In [ ]:
pgrins_full = sc.read_h5ad("Data/Projects/Keggoro/perturb_norm_pert_reduced.h5ad")
pgrins = {}
pgrins["Norman19"] = pgrins_full[pgrins_full.obs["PertNum"]<=75]
pgrins["Replogle22"] = pgrins_full[pgrins_full.obs["PertNum"]==-1 | pgrins_full.obs["PertNum"]>75]

exp = {}
exp["Norman19"] = sc.read_h5ad("Data/Experimental/Norman19/perturb_norm_subset_Keggoro.h5ad")
exp["Replogle22"] = sc.read_h5ad("Data/Experimental/Replogle22/perturb_norm_subset_Keggoro.h5ad")

perts = {}
for key, value in exp.items():
    perts[key] = list(value.obs["perturbation"].unique()).remove("ctrl")

# Visual comparison:

In [ ]:
# visualization method: heatmap

# Coexpression patterns:

## Coexpression graph?

# Shared DEGs:

## Get genes:

In [ ]:
# Returns a dict of dicts: each dataset has a dict with perturbations as keys and the DEGs for these perturbations as values
def get_degs(adata_dict):
    degs = {}
    for name in names:
        sc.tl.rank_genes_groups(adata_dict[name],groupby="perturbation",reference="ctrl",layer="log1p")
        degs[name] = {}
        for pert in perts[name]:
            degs[name][pert] = adata_dict[name].uns["rank_genes_groups"]["names"][pert][adata_dict[name].uns["rank_genes_groups"]["pvals_adj"][pert] < 0.05]
    return degs

In [ ]:
# Get genes whose mean expression value in a certain perturbation is higher than the mean of means
def get_on_genes(adata_dict):
    on_genes = {}
    for name in names:
        on_genes[name] = {}
        for pert in perts[name]:
            pert_mean = np.mean(adata_dict[name][adata_dict[name].obs["perturbation"]==pert].layers["log1p"],axis=0)
            on_genes[name][pert] = adata_dict[name].var_names[pert_mean>np.mean(pert_mean)]
    return on_genes

In [ ]:
# For negative control: get random number of genes equal to no. of genes per pert in pGRiNS
def get_rand_dict(gene_dict):
    dict_rand = {}
    for name in names:
        dict_rand[name] = {}
        for pert in perts[name]:
            dict_rand[name][pert] = choice(list(pgrins[name].var_names),size=len(gene_dict[name][pert]))
    return dict_rand

In [ ]:
names = ["Norman19","Replogle22"]
models = ["pGRiNS", "Random"]
tests = ["Common DEGs", "Common highly expressed genes"]

In [ ]:
exp_dict = {} # Has structure test -> name -> pert
model_dict = {} # Has structure test -> model -> name -> pert

exp_dict[tests[0]] = get_degs(exp)
exp_dict[tests[1]] = get_on_genes(exp)

model_dict[tests[0]] = {}
model_dict[tests[1]] = {}
model_dict[tests[0]][models[0]] = get_degs(pgrins)
model_dict[tests[1]][models[0]] = get_on_genes(pgrins)
model_dict[tests[0]][models[1]] = get_rand_dict(model_dict[tests[0]][models[0]])
model_dict[tests[1]][models[1]] = get_rand_dict(model_dict[tests[1]][models[0]])

## Chi squared:

In [ ]:
chi2_res_all = {} # Has structure test -> name -> model (list of pert_score)
for test in tests:
    chi2_res_all[test] = {}
    for name in names:
        chi2_res_all[test][name] = {}
        for model in models:
            chi2_res_all[test][name][model] = []
            for pert in perts[name]:
                degs_exp_set = set(exp_dict[test][name][pert])
                degs_model_set = set(model_dict[model][name][pert])
                all_genes = set(exp[name].var_names)

                cont_matrix = np.array([[len(degs_exp_set & degs_pgrins_set),len(degs_pgrins_set - degs_exp_set)],[len(degs_exp_set - degs_pgrins_set),len(all_genes - (degs_exp_set|degs_pgrins_set))]])
                res = chi2_contingency(cont_matrix)
                chi2_res_all[test][name][model].append(res.pvalue)

In [ ]:
# plot chi squared pval hist for perts
# for comparison: randomly assign genes as DEGs and calculate chi squared stat between that and exp data
# then do t test (or wilcoxon or sth) between distributions

In [ ]:
# How many genes are ON/OFF in both?

# Compare using metrics:

## Interpolated mean:

In [ ]:
# plot hist of distance between interpolated µ and each pert mean

In [ ]:
# or: across perts: plot mean MSE between adata_mean and adata_cell, and adata_mean and pgrins_cell

## Weighted metrics:

In [ ]:
# use WMSE, WR2, etc. to compare µ_c,exp and each µ_p,syn to GT of pert

# For weights calculation of interpolated duplicate etc. in GRiNS: is the order of pval_adj or t score alphabetically? Or are they reordered according to pval?